![IITIS](pictures/logoIITISduze.png)

# Wykorzystanie procesorów graficznych (GPU) w algorytmach heurystycznych

wykorzystamy paczkę numba (można też użyć CuPy wy bezpośrednio użwyać CUDA)

In [10]:
# funkcje pomocniczne

import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm
from typing import Optional

test_pegasus = os.path.join("instancje", "Pegasus", "P4_CBFM-P.txt")  # E = -469.0
full_pegasus = os.path.join("instancje", "Pegasus", "P16_CBFM-P.txt")  # E = -12772.0



def read_instance(path: os.PathLike):
    df = pd.read_csv(path, sep=" ", header=None, comment="#", names=["i", "j", "value"])

    n = max(df[["i", "j"]].max())
    h = np.zeros(n)
    J = np.zeros((n, n))
    
    for row in df.itertuples():
        if row.i == row.j:
            h[row.i - 1] = row.value
        elif row.i > row.j:
            J[row.j - 1, row.i - 1] = row.value  # by zachować górnotrójkątność
        else:
            J[row.i - 1, row.j - 1] = row.value
            
    return J, h


def dwave_conv_to_minus_half_convention(J: np.ndarray, h: np.ndarray):
    n = len(h)
    herminian_matrix = np.zeros((n, n))

    # de facto wyciągamy -1/2 przed macierz i zamieniamy ją na hermitowską
    for i in range(n):
        for j in range(i + 1, n):
            J_ij = J[i, j]
            herminian_matrix[i, j] = -J_ij
            herminian_matrix[j, i] = -J_ij

    x = np.random.choice([-1, 1], size=n)
    assert np.array_equal(-2 * x @ J @ x.T, x @ herminian_matrix @ x.T)
    assert np.array_equal(herminian_matrix.T, herminian_matrix)  # wszystkie macierze są rzeczywiste

    new_external_fields = -1 * h
    return herminian_matrix, new_external_fields


In [1]:
# Parrarel annealing
# konwencja: J górnotrójkątna, plusy

def calculate_gradient_pararell(J: np.ndarray, h: np.ndarray, x: np.ndarray, state: np.ndarray, 
lambda_t: float) -> np.ndarray:
    _, num_trajectories = x.shape
    return J @ state + J.T @ state + np.stack([h] * num_trajectories, axis=1) + lambda_t * x


def calculate_energy_parrarel(J: np.ndarray, h: np.ndarray, state: np.ndarray, convention: str = "PLUS"):
    n, _ = J.shape
    if convention == "PLUS":
        # Zakładamy, że J jest górnotrójkątna z zerami na przekątnych
        return np.sum(state * ( J @ state + h.reshape(n, 1)), axis=0)
    else:
        return np.sum(state * ( -1/2 * J @ state - h.reshape(n, 1)), axis=0)


def get_best_solution(states, energies):
    best_energy = np.min(energies)
    best_energy_index = np.argmin(energies)
    best_state = states[:, best_energy_index]  # Stany są w kolumnach
    return best_state, best_energy


def parrarel_annealing_multiple_trajectores(J, h, step_size: float, lambda_t_max: float, num_steps: int, num_trajectories: int,
                       schedule: Optional[np.ndarray] = None):
    n = len(h)
    x = np.zeros((n, num_trajectories))  # stan podstawowy dla H_innit = sum(x**2)
    momentum = np.zeros((n, num_trajectories))
    state = np.random.choice([-1, 1], size=(n, num_trajectories))  # losowy stan początkowy

    if schedule is None:
        schedule = np.linspace(lambda_t_max, 0, num=num_steps)

    for k in tqdm(range(num_steps), desc="wyżarzanie równoległe"):
        lambda_t = schedule[k]
        gradient = calculate_gradient_pararell(J, h, x, state, lambda_t)
        momentum = (1 - step_size) * momentum - step_size * gradient
        momentum = np.clip(momentum, -1, 1)
        x += momentum
        x = np.clip(x, -1, 1)
        state = np.sign(x)

    return state, calculate_energy_parrarel(J, h, state, num_trajectories)

In [19]:
import cupy as cp

def calculate_gradient_gpu_naive(J, h, x, state, lambda_t):
    gradient = cp.matmul(cp.multiply(-1, J), state) -  h + cp.multiply(lambda_t, x)
    return gradient

def calculate_energy_gpu_naive(J: np.ndarray, h: np.ndarray, state: np.ndarray):
    # Zakładamy, że J jest hermitowska z czynnikiem 1/2
    n, _ = J.shape
    A = cp.multiply(-1/2, J)
    B = cp.matmul(A, state) - h.reshape(n, 1)
    C = cp.multiply(state, B)
    return cp.sum(C, axis=0)



def parrarel_annealing_gpu_naive(J, h, step_size: float, lambda_t_max: float, num_steps: int, num_trajectories: int,
                       schedule: Optional[np.ndarray] = None, dtype = None):
    n = len(h)
    x = cp.zeros((n, num_trajectories))  # stan podstawowy dla H_innit = sum(x**2)
    momentum = cp.zeros((n, num_trajectories))
    state = cp.random.choice([-1, 1], size=(n, num_trajectories))  # losowy stan początkowy

    h_parrarel = cp.stack([h] * num_trajectories, axis=1)
    beta = cp.float32(1 - step_size)

    if schedule is None:
        schedule = cp.linspace(lambda_t_max, 0, num=num_steps)

    for k in tqdm(range(num_steps), desc="wyżarzanie równoległe GPU"):
        lambda_t = schedule[k]
        gradient = calculate_gradient_gpu_naive(J, h_parrarel, x, state, lambda_t)
        momentum = cp.multiply(beta, momentum) - cp.multiply(step_size, gradient)
        momentum = cp.clip(momentum, -1, 1)
        x = cp.clip(x + momentum, -1, 1)  
        state = cp.sign(x)


    return state, calculate_energy_gpu_naive(J, h, state)


J, h = read_instance(test_pegasus)
J, h = dwave_conv_to_minus_half_convention(J, h)
J = cp.asarray(J)
h = cp.asarray(h)


states, energies = parrarel_annealing_gpu_naive(J, h, step_size=0.01, lambda_t_max=10, num_steps=1000, num_trajectories=32)
print(min(energies))

wyżarzanie równoległe GPU: 100%|██████████| 1000/1000 [00:00<00:00, 2868.03it/s]

-469.0


In [ ]:
# test na pełnym pegazie
J, h = read_instance(full_pegasus)
J = cp.asarray(J)
h = cp.asarray(h)

states, energies = parrarel_annealing_gpu_naive(J, h, step_size=0.01, lambda_t_max=10, num_steps=10000, num_trajectories=2**10)
print(min(energies))

## Ilustracja rozłożenia w pamięci

In [ ]:
# własny kernel

from numba import cuda, float64, float32, int8
import numpy as np
import cupy as cp
# num_trajectories = 32

# J, h = read_instance(test_pegasus)
# J, h = dwave_conv_to_minus_half_convention(J, h)
# N = len(h)
# x = np.zeros((N, num_trajectories))  
# momentum = np.zeros((N, num_trajectories))
# state = np.random.choice([-1, 1], size=(N, num_trajectories))

# # move to GPU

# J = cuda.to_device(J)
# x = cuda.to_device(x)
# TPB = 16


@cuda.jit
def calculate_gradient_gpu(A, h, x, lambda_t, G):

    # dane dostępne dla każdego wątku, ograniczona ilość pamięci
    sX_col = cuda.shared.array(shape=threadsperblock, dtype=dtype)   
    sH = cuda.shared.array(shape=threadsperblock, dtype=dtype)
    sA = cuda.shared.array(shape=threadsperblock, dtype=dtype)


    ti = cuda.threadIdx.x  # pojedyńczy element w kolumnie
    col = cuda.blockIdx.x  # każdy blok zajmuje się jedną kolumną (trajektorią)
    bpg = cuda.gridDim.y  # ilośc bloków w jednej kolumnie

    # wczytujemy kawałek danych i wykonujemy operacje
    for b in range(bpg):
        sX_col[ti] = 0
        sH[ti] = 0
        sA[ti] = 0

        global_row = ti + b * threadsperblock
        
        if global_row < x.shape[0]:
            sX_col[ti] = x[global_row, col]
            sH[ti] = h[global_row]
            sA[ti] = A[global_row, col]
        cuda.syncthreads()

        G[global_row, col] = -1 * sA[ti] - sH[ti] + sX_col[ti] * lambda_t 
    



N = 5000
M = 2**10
dtype = float32
h = cuda.to_device(np.ones(N))
x = cuda.to_device(np.ones((N, M)))

G = cuda.device_array_like(x)

state = cp.asarray(np.ones((N, M)))
J = cp.asarray(np.ones((N, N)))


threadsperblock = min(256, N)  # max 256 wątków w jedym bloku, jak mniej to musimy wziąść mniej
blockspergrid_x = M  # każdy blok zajmuje się trajektorią
blockspergrid_y = (N + threadsperblock - 1) // threadsperblock  # wystarczająca ilość bloków by pomieścić całą kolumnę 
blockspergrid = (blockspergrid_x, blockspergrid_y)


A = cp.matmul(J, state)  # nie wymyślajmy koła od nowa
calculate_gradient_gpu[blockspergrid, threadsperblock](A, h, x, 0.5, G)
print(G.copy_to_host())
print(G.copy_to_host().shape)

# 


256
(1024, 20)
[[-5000.5 -5000.5 -5000.5 ... -5000.5 -5000.5 -5000.5]
 [-5000.5 -5000.5 -5000.5 ... -5000.5 -5000.5 -5000.5]
 [-5000.5 -5000.5 -5000.5 ... -5000.5 -5000.5 -5000.5]
 ...
 [-5000.5 -5000.5 -5000.5 ... -5000.5 -5000.5 -5000.5]
 [-5000.5 -5000.5 -5000.5 ... -5000.5 -5000.5 -5000.5]
 [-5000.5 -5000.5 -5000.5 ... -5000.5 -5000.5 -5000.5]]
(5000, 1024)


In [29]:
def parrarel_annealing_gpu(J, h, step_size: float, lambda_t_max: float, num_steps: int, num_trajectories: int,
                       schedule: Optional[np.ndarray] = None, dtype = None):
    n = len(h)
    x = cp.zeros((n, num_trajectories))  # stan podstawowy dla H_innit = sum(x**2)
    momentum = cp.zeros((n, num_trajectories))
    state = cp.random.choice([-1, 1], size=(n, num_trajectories))  # losowy stan początkowy

    h_parrarel = cp.stack([h] * num_trajectories, axis=1)
    beta = cp.float32(1 - step_size)

    threadsperblock = min(256, n)  # max 256 wątków w jedym bloku, jak mniej to musimy wziąść mniej
    blockspergrid_x = num_trajectories  # każdy blok zajmuje się trajektorią
    blockspergrid_y = (n + threadsperblock - 1) // threadsperblock  # wystarczająca ilość bloków by pomieścić całą kolumnę 
    blockspergrid = (blockspergrid_x, blockspergrid_y)

    if schedule is None:
        schedule = np.linspace(lambda_t_max, 0, num=num_steps)

    G = cuda.device_array_like(x)
    # do poprawy, może napisz własne mnożenie macierzy
    for k in tqdm(range(num_steps), desc="wyżarzanie równoległe GPU"):
        lambda_t = schedule[k]
        
        A = cp.matmul(J, state)  # nie wymyślajmy koła od nowa
        calculate_gradient_gpu[blockspergrid, threadsperblock](A, h, x, lambda_t, G)
        gradient = cp.asarray(G)
        momentum = cp.multiply(beta, momentum) - cp.multiply(step_size, gradient)
        momentum = cp.clip(momentum, -1, 1)
        x = cp.clip(x + momentum, -1, 1)  
        state = cp.sign(x)


    return state, calculate_energy_gpu_naive(J, h, state)

In [ ]:
J, h = read_instance(full_pegasus)
J, h = dwave_conv_to_minus_half_convention(J, h)

J = cp.asarray(J)
h = cp.asarray(h)

states, energy = parrarel_annealing_gpu(J, h, step_size=0.01, lambda_t_max=10, num_steps=10000, num_trajectories=2**10)
print(min(energy))

# 